[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Math/Linear_Algebra/Linear_Algebra.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Linear Algebra for Signals

The most-borrowed-from course in the curriculum: transforms are changes of basis, filters and networks are matrices, PCA and attention are eigen/SVD stories. Five sessions build exactly the linear algebra the other workshops silently assume — always with a signal in hand.

## 0. Introduction

The through-line: **a signal is a vector; every operation we care about is a matrix; understanding a matrix means finding the basis in which it is simple.**

## 1. Pre-requisites

[Intro to Python](../../Intro_Programming/Intro_Python/Intro_Python.ipynb) (NumPy). No prior linear algebra assumed; comfort with vectors as arrows helps.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 5 — *Vectors, Bases & Change of Basis* (~35 min)
**Goal:** treat signals as vectors; understand span, independence, and what a basis buys you.
**Feeds into:** Session 2 (projections).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Vectors, Bases & Change of Basis</b></summary>

**Timing (~35 min).** 10 min "a signal is a vector" · 10 min independence, span, basis · 15 min the two-basis demo and what it implies.

**Open with the sentence the whole workshop hangs on.** *A signal is a vector; every operation we care about is a matrix; understanding a matrix means finding the basis in which it is simple.* Say it at the start, then close each session by pointing at which clause was just cashed out. This workshop is the most borrowed-from in the curriculum, and students retain it far better when they know all five sessions are one argument rather than five topics.

**Make the identification concrete before the abstraction.** A 64-sample recording *is* a point in $\mathbb{R}^{64}$ — one coordinate per sample. Adding two signals, scaling one, mixing a bank of them: all vector arithmetic they already do. The abstraction is not adding anything; it is naming what they were already doing so the tools become available.

**Insist that independence has an operational meaning.** $\sum c_i v_i = 0 \Rightarrow$ all $c_i = 0$ says no reference signal is a mix of the others — nothing is redundant. Then the payoff: independence is exactly what makes coordinates **unique**. Without it, the same signal has many recipes and "the coordinates of $x$" is not a well-defined phrase. Students who see uniqueness as the point stop treating independence as a technicality.

**Then the fact that makes everything downstream cheap: orthonormality turns inversion into transposition.** If $B^TB = I$ then $c = B^Tx$ — coordinates are just inner products, one projection per basis vector, no linear solve. Say what this costs elsewhere: a general basis needs $O(N^3)$ to invert, an orthonormal one needs $O(N^2)$ to apply, and a *structured* orthonormal one like the DCT needs $O(N\log N)$. That chain is why the FFT matters and why the Fourier basis is the star of [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb).

**Have the room predict the demo before running it.** Same Gaussian bump, two bases. Ask how many nonzero coordinates each will need. The standard basis needs all 64 by construction — a spike basis has no way to say "smooth". The cosine basis needs **6** for 99.9% of the energy. Let the surprise land before explaining it.

**Then explain it, because "sparsity" is not magic.** A basis is a set of questions you ask a signal. The standard basis asks "what is the value at sample 37?", which a smooth bump answers 64 times with 64 different numbers. The cosine basis asks "how much slow wiggle, how much fast wiggle?", and a smooth bump answers "a lot of slow, none of the fast" — six numbers. **Sparsity is a statement about the match between a signal and a basis, never about the signal alone.** A cymbal crash is sparse in the standard basis and dense in the cosine basis; the roles reverse completely.

**Land the applications, since three later workshops depend on this one paragraph.** JPEG stores DCT coefficients and discards the small ones. [Compressed Sensing](../../Intro_DSP/Compressed_Sensing.ipynb) assumes such a sparsifying basis exists and asks how few measurements you need. The DFT is precisely this change of basis with complex exponentials. Every one of them is "change basis until the signal gets simple", which is the session's title read literally.
</details>

## 2. Signals Are Vectors

💡 **Intuition.** A sampled signal of length $N$ *is* a point in $\mathbb{R}^N$ — each sample is a coordinate. Adding signals, scaling signals, mixing signals: all vector operations. A **basis** is a set of $N$ independent reference signals; *any* signal is a unique recipe (coordinate vector) over them. Changing basis doesn't change the signal — it changes which recipe you read.

**Definitions.** Vectors $\{v_1, \dots, v_k\}$ are *linearly independent* if $\sum_i c_i v_i = 0$ forces all $c_i = 0$ (no vector is a mix of the others). Their *span* is everything expressible as such mixes. A *basis* of $\mathbb{R}^N$ = $N$ independent vectors; then every $x$ has unique coordinates. If the basis vectors are orthonormal ($B^T B = I$), coordinates are just inner products: $c = B^T x$.

In [2]:
N = 64
t = np.arange(N)

# Basis 1: the standard basis (each vector = one sample spike)
# Basis 2: cosines of increasing frequency (a mini-DCT) — orthonormal
k = np.arange(N)
B = np.cos(np.pi * (t[:, None] + 0.5) * k[None, :] / N)
B[:, 0] *= 1 / np.sqrt(2)
B *= np.sqrt(2 / N)
print("orthonormal check ‖BᵀB − I‖ =", np.abs(B.T @ B - np.eye(N)).max().round(12))

orthonormal check ‖BᵀB − I‖ = 0.0


**What just happened.** $\|B^TB - I\|_\infty = 0.0$ to twelve decimal places — the 64 cosines really are mutually orthogonal and unit-norm, so $B$ is an orthonormal basis and not merely a set of 64 vectors that happen to span.

**That zero is worth pausing on, because it is what makes the rest of the session cheap.** For a general basis, finding coordinates means solving $Bc = x$: an $O(N^3)$ factorisation, and a numerically delicate one if the basis is ill-conditioned. For an orthonormal basis, $B^{-1} = B^T$, so $c = B^Tx$ — one matrix-vector product, $O(N^2)$, no solve, no conditioning worries. **Inversion collapses into transposition**, and every projection formula later in this notebook simplifies the same way.

**The three lines of scaling are what earn the orthonormality, and each does a specific job.** The $t + \tfrac12$ half-sample offset makes this the DCT-II rather than a naive sampled cosine, which is what makes the inner products vanish exactly rather than approximately. The $1/\sqrt2$ on column 0 corrects the DC vector, whose entries are all equal and whose norm therefore differs from the rest. The global $\sqrt{2/N}$ normalises everything to unit length. Drop any one and the check prints something nonzero.

**And note that the cost story does not end at $O(N^2)$.** This basis is not just orthonormal, it is *structured*: the DCT can be applied in $O(N\log N)$ by an FFT-based algorithm, never forming $B$ at all. Explicit matrix, orthonormal matrix, structured orthonormal matrix — $O(N^3)$, $O(N^2)$, $O(N\log N)$. That progression is the entire reason JPEG and the FFT are practical, and it is a property of the *basis*, not of any signal.

**One honest caveat about the printed zero.** It is `.round(12)` of a floating-point maximum, so it means "below $5\times10^{-13}$", not literally zero. The true orthogonality is exact in real arithmetic; what is shown is that double precision has not degraded it. Reassuring, and not the same claim.

In [3]:
# The SAME smooth signal, read in both bases
x = np.exp(-0.5 * ((t - 24) / 8) ** 2)          # a smooth bump
c = B.T @ x                                      # coordinates in the cosine basis

fig, axes = plt.subplots(1, 2, figsize=(9, 2.6))
axes[0].plot(x); axes[0].set_title("standard-basis coordinates (the samples)")
axes[1].stem(c); axes[1].set_title("cosine-basis coordinates: nearly all zero!")
plt.tight_layout(); plt.show()
print(f"samples needed to capture 99.9% energy: standard {N}, cosine {int((np.sort(c**2)[::-1].cumsum() / (c**2).sum() < 0.999).sum()) + 1}")

samples needed to capture 99.9% energy: standard 64, cosine 6


/tmp/ipykernel_2010461/3565037298.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** One signal, two descriptions. The left panel needs all **64** numbers — the standard basis has no vocabulary for "smooth", so it reports the bump one sample at a time. The right panel is almost entirely zero: **6 coefficients carry 99.9% of the energy**, a better than 10× compression of the same information with no loss worth measuring.

**Nothing about the signal changed.** Same vector, same energy, same everything — $\|c\| = \|x\|$ exactly, because $B$ is orthonormal and rotations preserve length. What changed is the **question being asked**. The standard basis asks "what is the value at sample 37?" and a smooth bump answers 64 times with 64 different numbers. The cosine basis asks "how much slow wiggle? how much fast?" and the same bump answers "a lot of slow, essentially no fast" — six numbers.

**So sparsity is a property of the *pairing*, not of the signal.** This is the single most important idea in the session and students routinely mis-file it as "smooth signals are sparse." Take a cymbal crash — one loud sample, silence around it. In the standard basis it is 1-sparse. In the cosine basis it is *dense*: a spike needs every frequency, all contributing equally. The roles reverse completely. Neither signal is intrinsically simple; each is simple in the basis built to describe it.

**The mechanism is worth stating precisely.** A basis vector "matches" a signal to the extent that their inner product is large, and $c_k = \langle b_k, x\rangle$ is exactly that. A Gaussian bump of width 8 samples has almost no correlation with a cosine oscillating every 3 samples, so those coefficients are near zero — not by luck, but because the signal genuinely contains no such structure to report.

**This one plot is the premise of three later workshops, so name them.** JPEG is this demo applied to $8\times8$ image blocks: transform, keep the big coefficients, discard the rest. [Compressed Sensing](../../Intro_DSP/Compressed_Sensing.ipynb) starts from the assumption that a sparsifying basis exists and asks how few measurements you need to recover the signal — the answer scales with the 6, not the 64. Denoising in the next session is the same move: noise is dense in every basis, so the coefficients where the signal lives are exactly where the signal-to-noise ratio is high.

**One caveat on reading "nearly all zero".** The threshold is 99.9% of *energy*, which is a squared measure and therefore forgiving — coefficients at 3% of the peak contribute 0.09% of the energy and get counted as negligible. Reconstruct from 6 and you will see a visually perfect bump with a small residual ripple. That is a real approximation, not an identity, and the honest claim is "6 coefficients suffice for this tolerance", not "the signal has 6 nonzero coordinates."

That collapse — smooth signal, sparse cosine recipe — is why JPEG exists, and it's the entire premise of [Compressed Sensing](../../Intro_DSP/Compressed_Sensing.ipynb). The DFT of [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) is exactly a change to a complex-exponential basis.

---
### 🕐 Session 2 of 5 — *Projections & Least Squares* (~35 min)
**Goal:** project onto subspaces; derive the normal equations; fit models as projections.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (eigendecomposition).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Projections & Least Squares</b></summary>

**Timing (~35 min).** 8 min the shadow picture and orthogonality of the residual · 10 min deriving the normal equations · 10 min the denoising demo · 7 min least squares as model fitting.

**Start geometrically and refuse to write a formula for the first ten minutes.** Your signal is not in the subspace you can build from. The best you can do is its **shadow** — the closest point of that subspace. Then ask the room *why* the residual must be perpendicular: because if it had any component lying inside the subspace, you could slide along that component and get closer. Optimality and orthogonality are the same statement, and once a student sees that, the normal equations write themselves.

**Derive $A^T(x - Ac) = 0$ from that sentence, not from calculus.** "The residual is orthogonal to every column of $A$" is literally $A^T r = 0$. Expand and you have $A^TAc = A^Tx$. Calculus gives the same answer in Session 5 via $\nabla_c J = 2A^TAc - 2A^Tx$, and it is worth promising that now so the room sees the two routes converge.

**Spend a minute on $P^2 = P$, because it is the definition of a projector and it is intuitive.** Once you are in the subspace, projecting again does nothing — the shadow of a shadow is the shadow. Then note that $P$ is also symmetric here, which is what makes it an *orthogonal* projector rather than a merely oblique one.

**Flag the simplification the demo uses, or students will not follow the code.** With orthonormal columns $A^TA = I$, so $P = AA^T$ and the whole $(A^TA)^{-1}$ disappears. This is Session 1's payoff arriving on schedule: orthonormality turns a matrix inverse into a transpose. It is also why nobody computes $(A^TA)^{-1}$ in practice — the numerically sound route is QR or SVD, which is exactly what `lstsq` does under the hood.

**Set up the denoising demo as a prediction question.** Project a noisy signal onto the 8-dimensional smooth subspace. Ask what happens to the noise. The clean answer: noise is spread evenly over all 64 dimensions, so keeping 8 keeps $8/64$ of its energy — a $\sqrt8 \approx 2.8\times$ reduction in RMS. The measured improvement is only **2×**, and the gap is the lesson. The true signal is not exactly inside the 8-D subspace either, so throwing away dimensions throws away a little signal too. That is the **bias–variance tradeoff**, visible in one number, and it is the same tradeoff that governs model order everywhere in the curriculum.

**On the least-squares fit, insist that the interesting coefficient is the one that should be zero.** The design matrix includes a $\cos$ term the data does not contain, and the fit returns 0.029 for it. Ask whether that is "close enough to zero" — the right answer is not a judgement call but a computation: the standard error is $\sigma\sqrt{2/m} = 0.03$, so the estimate is one standard error from zero and entirely consistent with the truth. **"Is this coefficient zero?" is a question about the noise level, never about the digits.** Rooms with a statistics background will appreciate that this is the whole content of a $t$-test.

**Close on the connection students will meet again within two workshops.** The Wiener solution $\mathbf{w}_o = R^{-1}\mathbf{p}$ in [Adaptive Filtering](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb) *is* $A^TAc = A^Tx$ with $R = A^TA$ and $\mathbf{p} = A^Tx$. Nothing new is happening there; it is this session's geometry with expectations in place of sums.
</details>

## 3. Projection

💡 **Intuition.** If your signal can't be built exactly from the vectors you have, the best you can do is the **shadow**: the point of the subspace closest to the signal. The defining property is that the leftover (residual) is *orthogonal* to the subspace — if it weren't, you could slide within the subspace to shrink the error. Least squares is nothing but this geometry with data.

**Normal equations.** To minimize $\|Ac - x\|^2$ over recipes $c$ (columns of $A$ = your building blocks): the residual must satisfy $A^T (x - Ac) = 0$, giving

$$A^T A \, c = A^T x \qquad \Rightarrow \qquad \hat{x} = A (A^T A)^{-1} A^T x$$

The matrix $P = A(A^TA)^{-1}A^T$ is the *projector* onto the column space: $P^2 = P$ (projecting twice changes nothing).

In [4]:
# Denoise by projection: noisy smooth signal onto the first 8 cosine basis vectors
x_clean = np.exp(-0.5 * ((t - 24) / 8) ** 2)
x_noisy = x_clean + 0.1 * rng.standard_normal(N)

A = B[:, :8]                                    # low-frequency subspace
xhat = A @ (A.T @ x_noisy)                      # orthonormal columns ⇒ projector is AAᵀ

plt.figure(figsize=(8, 2.6))
plt.plot(x_noisy, alpha=0.5, label="noisy")
plt.plot(xhat, linewidth=2, label="projection onto 8-D smooth subspace")
plt.plot(x_clean, "k--", linewidth=1, label="truth")
plt.legend(); plt.title("Denoising = projection")
plt.tight_layout(); plt.show()
print(f"noise RMSE {np.std(x_noisy - x_clean):.4f} → projected RMSE {np.std(xhat - x_clean):.4f}")

noise RMSE 0.0912 → projected RMSE 0.0453


/tmp/ipykernel_2010461/1909784845.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Projecting the noisy signal onto the 8-dimensional smooth subspace cut the error from **0.0912 to 0.0453** — a factor of 2.01. The projected curve tracks the dashed truth closely while the raw noisy trace jitters around it. No filter was designed, no cutoff frequency chosen: the entire operation was $\hat x = AA^Tx$, one matrix product.

**But check the 2× against what theory predicts, because they do not match.** White noise spreads its energy evenly across all 64 orthonormal directions. Keeping 8 of them keeps $8/64 = 1/8$ of the noise energy, so the noise RMS should fall by $\sqrt8 = 2.83\times$, to about **0.032**. We measured 0.0453. The projection is doing *worse* than the noise argument allows.

**The gap is signal, not noise — and that is the lesson.** The clean Gaussian bump is not exactly inside the 8-D subspace; from Session 1 we know it needs 6 coefficients for 99.9% of its energy, so a small tail lives in the discarded directions. Truncating at 8 throws that tail away. Decomposing the measured error:

$$\underbrace{0.0453^2}_{\text{total}} \;\approx\; \underbrace{0.032^2}_{\text{surviving noise (variance)}} \;+\; \underbrace{0.032^2}_{\text{discarded signal (bias)}}$$

The two contributions are almost exactly equal here. That is the **bias–variance tradeoff**, and this cell measures both halves of it in one number.

**Which makes the subspace dimension a genuine design knob rather than a free parameter.** Shrink to 4 dimensions and the noise falls further ($\sqrt{16} = 4\times$) while the bias grows — the bump starts to visibly flatten. Grow to 32 and the bump is reproduced perfectly while only $\sqrt2$ of noise reduction survives. There is an optimum, it depends on the SNR and on how sparse the signal actually is, and **no choice makes both terms small at once**. Every model-order, filter-length, and rank-selection decision in the rest of the curriculum is this same curve.

**Note what makes the code so short, since it is Session 1 arriving on schedule.** The general projector is $P = A(A^TA)^{-1}A^T$. Because the DCT columns are orthonormal, $A^TA = I$ and it collapses to $AA^T$ — no inverse, no solve. The line `xhat = A @ (A.T @ x_noisy)` is a full least-squares fit, and the parenthesisation matters: it costs two $O(N \cdot 8)$ products rather than forming a $64\times64$ matrix.

**Finally, name what this is in signal-processing language.** Keeping the low-frequency DCT coefficients and zeroing the rest *is* an ideal low-pass filter — designed here without ever mentioning frequency response, cutoff, or convolution, purely as "project onto the subspace I believe the signal lives in." That reframing is what makes projection portable: swap the DCT columns for any dictionary matching your signal and the same three lines denoise it.

In [5]:
# Least squares as model fitting: recover a line + sine trend from noisy data
m = 200
tt = np.linspace(0, 1, m)
y = 2.0 + 1.5 * tt + 0.8 * np.sin(2 * np.pi * 3 * tt) + 0.3 * rng.standard_normal(m)

A = np.stack([np.ones(m), tt, np.sin(2 * np.pi * 3 * tt), np.cos(2 * np.pi * 3 * tt)], axis=1)
c, *_ = np.linalg.lstsq(A, y, rcond=None)
print("recovered coefficients:", c.round(3), " (truth: [2, 1.5, 0.8, 0])")

recovered coefficients: [2.022 1.439 0.805 0.029]  (truth: [2, 1.5, 0.8, 0])


**What just happened.** Four coefficients recovered from 200 noisy points:

| term | truth | fit | error |
|---|---|---|---|
| constant | 2.0 | 2.022 | +0.022 |
| slope | 1.5 | 1.439 | −0.061 |
| $\sin$ | 0.8 | 0.805 | +0.005 |
| $\cos$ | **0** | 0.029 | +0.029 |

**The right question is not "are these close?" but "are they within the noise?"** With $\sigma = 0.3$ and $m = 200$, the standard error on the sinusoid coefficients is $\sigma\sqrt{2/m} = 0.030$, and on the slope roughly $\sigma/(\sqrt m \cdot \mathrm{std}(t)) \approx 0.074$. Every error above is inside one standard error. The slope being off by 0.061 looks like the worst result in the table and is in fact the most ordinary — it has the largest error bar.

**The coefficient worth the room's attention is the $\cos$ term.** We deliberately fitted a component the data does not contain, and least squares returned 0.029 rather than 0.000. It could not have returned zero: the noise has some accidental correlation with $\cos(6\pi t)$, and the fit dutifully reports it. **A coefficient that should be zero comes back at roughly one standard error, every time.** Judging it by digits ("0.029 is small") is guesswork; judging it against $\sigma\sqrt{2/m} = 0.030$ is a measurement. That comparison is exactly what a $t$-test computes.

**So "the model has too many terms" has a precise cost, and it is visible here.** Each extra column in $A$ gives the noise one more direction to project into. Add ten spurious basis functions and you will get ten spurious coefficients, each around $0.03$, each looking like a small real effect. That is overfitting stated in the language of this session: **a bigger subspace captures more noise**, the same tradeoff the denoising cell measured, now in coefficient space.

**Note that this fit is genuinely the same operation as the denoising one.** There, $A$ held DCT columns and the projection was a filter; here $A$ holds $\{1, t, \sin, \cos\}$ and the projection is a regression. Identical geometry — find the point of $\mathrm{col}(A)$ nearest to the data, make the residual orthogonal to every column. Only the choice of building blocks changed, and that choice is the entire modelling decision.

**One implementation remark.** `np.linalg.lstsq` does not form $(A^TA)^{-1}$; it solves via SVD, which is numerically stable even when the columns are nearly dependent. Forming the normal equations explicitly squares the condition number — fine here, dangerous with polynomial or closely-spaced sinusoidal bases. The formula in the text is for understanding; this call is what you ship.

The Wiener solution $\mathbf{w}_o = R^{-1}\mathbf{p}$ in [Adaptive Filtering](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb) *is* a normal equation — now you know why it looks the way it does.

---
### 🕐 Session 3 of 5 — *Eigendecomposition* (~35 min)
**Goal:** find the directions a matrix merely stretches; diagonalize symmetric matrices.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (SVD).

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Eigendecomposition</b></summary>

**Timing (~35 min).** 10 min eigenvectors as the matrix's own axes · 8 min the spectral theorem and its consequences · 10 min power iteration · 7 min covariance eigenvectors and the LMS connection.

**Motivate before defining.** Most vectors get rotated *and* stretched by a matrix, which is why matrices are hard to reason about. Eigenvectors are the directions where rotation does not happen — $Av = \lambda v$, pure stretch. In the basis of those directions the matrix *is* a list of numbers, and "understanding a matrix means finding the basis in which it is simple" (Session 1's through-line) is answered literally.

**Give the room the three consequences that make it worth the trouble, concretely.** $S^k = Q\Lambda^kQ^T$ turns matrix powers into scalar powers — stability of a recursion becomes "is every $|\lambda| < 1$?". $e^{S} = Qe^{\Lambda}Q^T$ turns matrix exponentials into scalar ones, which is how every linear ODE is solved. And $x^TSx = \sum_i \lambda_i(q_i^Tx)^2$ turns a quadratic form into a weighted sum of squares, which is why positive-definite means "all $\lambda_i > 0$" and why the loss surface in the [ANN workshop](../../Intro_Mach_Learn/Intro_ANN/Intro_ANN.ipynb) is a bowl rather than a saddle.

**The spectral theorem deserves to be called the jackpot, and it is worth saying what could have gone wrong.** General matrices can have complex eigenvalues, non-orthogonal eigenvectors, or too few eigenvectors to form a basis at all — $\begin{psmallmatrix}0&1\\0&0\end{psmallmatrix}$ has a single eigenvector and is not diagonalisable. Symmetric matrices suffer none of this: real eigenvalues, orthonormal eigenvectors, always a full basis. Since covariance matrices, Gram matrices, Hessians, and graph Laplacians are all symmetric, this covers essentially everything the curriculum needs.

**Power iteration is four lines and rewards being derived rather than shown.** Write any $v$ in the eigenbasis, apply $S$ repeatedly, and each component scales by its own $\lambda^k$ — so the largest eigenvalue's component dominates by a factor $(\lambda_2/\lambda_1)^k$. The normalisation only prevents overflow. Ask what makes it slow: the *ratio*, not the size, of the top two eigenvalues. Then note that PageRank, spectral clustering, and the classic top-PCA implementations are all this loop.

**Have the room predict the covariance demo before running it.** Points drawn from an anisotropic Gaussian, arrows drawn along the eigenvectors. The arrows land on the visible axes of the cloud, which is not a coincidence but the definition: $q^TCq$ is the variance in direction $q$, and the eigenvectors are its extremes. Then note the numbers — eigenvalues $[0.18, 5.09]$, a 28:1 variance ratio, meaning one direction carries 97% of the spread. That is dimensionality reduction discovered rather than imposed.

**Close on the LMS connection, since it converts an abstract bound into a picture.** The convergence limit $\mu < 2/\lambda_{\max}$ in [Adaptive Filtering](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb) is now readable: the steepest axis of the bowl sets the speed limit, because a step size that is stable along the shallow axis will diverge along the steep one. And the *rate* is governed by the eigenvalue spread $\lambda_{\max}/\lambda_{\min}$ — the same 28:1 ratio in the plot would mean 28× more iterations along the cramped direction. Students who have already met LMS usually find this the most clarifying five minutes of the workshop.
</details>

## 4. Eigenvectors

💡 **Intuition.** Most vectors get rotated *and* stretched by a matrix. Eigenvectors are the exceptions — directions the matrix only **stretches** ($Av = \lambda v$). They are the matrix's natural axes: in the eigenbasis, the matrix becomes a diagonal list of stretch factors, and hard things (powers, exponentials, stability) become arithmetic. For symmetric matrices the spectral theorem is the jackpot: eigenvalues real, eigenvectors orthonormal — a *perfect* basis.

**Spectral theorem.** Symmetric $S = S^T \in \mathbb{R}^{N\times N}$ has $S = Q \Lambda Q^T$ with $Q$ orthonormal, $\Lambda$ real diagonal. Consequences: $S^k = Q\Lambda^k Q^T$; quadratic form $x^T S x = \sum_i \lambda_i (q_i^T x)^2$; positive-definite ⇔ all $\lambda_i > 0$ (the loss *bowl* of the [ANN workshop](../../Intro_Mach_Learn/Intro_ANN/Intro_ANN.ipynb) is such a form).

In [6]:
# Power iteration: the eigenvector algorithm you can write in four lines
S = np.array([[2.0, 1.0], [1.0, 3.0]])
v = rng.standard_normal(2)
for _ in range(50):
    v = S @ v
    v /= np.linalg.norm(v)
lam = v @ S @ v
w, V = np.linalg.eigh(S)
print(f"power iteration: λ={lam:.6f}  v={np.round(v, 4)}")
print(f"numpy eigh:      λ={w[-1]:.6f}  v={np.round(V[:, -1], 4)}   (sign may flip)")

power iteration: λ=3.618034  v=[0.5257 0.8507]
numpy eigh:      λ=3.618034  v=[0.5257 0.8507]   (sign may flip)


**What just happened.** Fifty multiply-and-normalise steps, starting from a random vector, and the result matches LAPACK's `eigh` to six decimal places: $\lambda = 3.618034$, $v = [0.5257, 0.8507]$. A four-line loop with no factorisation, no solve, and no library call reproduced a professional eigensolver.

**Check the answer against theory rather than against the library.** For $S = \begin{psmallmatrix}2&1\\1&3\end{psmallmatrix}$ the characteristic polynomial is $\lambda^2 - 5\lambda + 5$, giving $\lambda = \frac{5 \pm \sqrt5}{2}$ — exactly $3.618034$ and $1.381966$. The printed value is the closed form, not a coincidence of two implementations agreeing. And the eigenvector components satisfy $0.8507/0.5257 = 1.618 = \varphi$, the golden ratio, which falls out of $\lambda_1 = \varphi + 1 = \varphi^2$.

**Why it works is one line of algebra worth doing at the board.** Write the starting vector in the eigenbasis, $v = c_1q_1 + c_2q_2$. Then $S^kv = c_1\lambda_1^kq_1 + c_2\lambda_2^kq_2$, so the ratio of the two components evolves as $(\lambda_2/\lambda_1)^k$. The dominant direction wins by *exponential attrition*, not by anything clever. Normalising each step only prevents overflow; it does not affect the direction.

**So the convergence rate is set by the eigenvalue *ratio*, and that is the practical lesson.** Here $\lambda_2/\lambda_1 = 1.381966/3.618034 = 0.382$, so after 50 steps the contaminating component has shrunk by $0.382^{50} \approx 10^{-21}$ — far below double precision, which is why the match is exact to every digit printed. Take a matrix with $\lambda_2/\lambda_1 = 0.99$ and the same 50 steps buy a factor of only 0.6; you would need about 4,000 iterations for the same accuracy. **Well-separated spectra converge instantly, clustered ones crawl** — and that is exactly the failure mode diagnosed in [Manifold Optimization](../Optimization/Manifold_Optimization.ipynb), where a near-degenerate spectrum makes the same idea stall.

**Note the honest caveats, since the demo is deliberately friendly.** Power iteration finds only the **largest** eigenvalue, gives nothing about the others without deflation, and fails outright if the random start happens to be orthogonal to $q_1$ — an event of probability zero in exact arithmetic, and rescued by rounding noise in practice. Also, the sign is arbitrary: $-v$ is equally an eigenvector, which is why the printout warns about it. Any downstream code comparing eigenvectors must fix a sign convention or compare $|q^Tv|$.

**Finally, note that this loop is not a toy.** PageRank is power iteration on a web-link matrix. Spectral clustering, the classic top-$k$ PCA implementations, and Krylov methods like Lanczos all start from exactly this observation — that repeated multiplication amplifies the dominant direction. When the matrix is too large to factor or even to store, multiply-and-normalise is often the only method available.

In [7]:
# Eigenvectors of a covariance = the axes of the data cloud
X = rng.standard_normal((500, 2)) @ np.array([[2.0, 0.0], [1.2, 0.5]])
C = np.cov(X.T)
w, V = np.linalg.eigh(C)

plt.figure(figsize=(4.5, 4.5))
plt.scatter(*X.T, s=4, alpha=0.4)
for lam_i, vec in zip(w, V.T):
    plt.arrow(0, 0, *(2 * np.sqrt(lam_i) * vec), width=0.03, color="crimson")
plt.axis("equal"); plt.title("covariance eigenvectors = the cloud's own axes")
plt.tight_layout(); plt.show()
print("eigenvalues (variances along the axes):", w.round(2))

eigenvalues (variances along the axes): [0.18 5.09]


/tmp/ipykernel_2010461/423711697.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The two red arrows land on the visible axes of the point cloud — the long one along the direction the data spreads most, the short one perpendicular to it. The eigenvalues are $[0.18,\ 5.09]$: the variance along each axis, and the arrow lengths are $2\sqrt{\lambda_i}$, i.e. two standard deviations.

**The arrows were not fitted to the picture; they came out of a $2\times2$ matrix.** Nothing in the code looked at the shape of the cloud. `np.cov` reduced 500 points to four numbers, `eigh` decomposed those, and the resulting directions happen to be the ones your eye picks out. That agreement is the content of the demo: $q^TCq$ *is* the variance in direction $q$, so the eigenvectors — which extremise that quadratic form — are exactly the directions of most and least spread. The eye and the algebra are computing the same thing.

**Check the numbers against the generator, because they are close but not exact.** The data is $Z A$ with $A = \begin{psmallmatrix}2&0\\1.2&0.5\end{psmallmatrix}$, so the true covariance is $A^TA = \begin{psmallmatrix}5.44&0.6\\0.6&0.25\end{psmallmatrix}$, whose eigenvalues are $5.51$ and $0.18$. We measured $5.09$ and $0.18$. The top eigenvalue is about 8% low, and with $n = 500$ the relative standard error on an eigenvalue is roughly $\sqrt{2/n} = 6.3\%$ — so this is a little over one standard error, entirely ordinary sampling variation. **A covariance estimated from finite data is itself a random matrix**, which is the whole subject of [Random Matrix Theory](../Random_Matrix_Theory/Random_Matrix_Theory.ipynb).

**The ratio is the number worth remembering: 5.09/0.18 ≈ 28.** One direction carries 97% of the total variance. That means the cloud is, to a very good approximation, one-dimensional — and nobody told the algorithm to look for that. **PCA is dimensionality reduction discovered from the data rather than imposed on it**, and it is nothing more than what this cell already did: eigendecompose the covariance, keep the large eigenvalues.

**That same ratio is also a warning, in a different context.** If this matrix were the input correlation $R$ of an adaptive filter, the eigenvalue spread 28:1 is exactly what sets LMS convergence in [Adaptive Filtering](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb). The step size must satisfy $\mu < 2/\lambda_{\max}$ — the *steep* axis sets the stability limit — while the error along the *shallow* axis decays at a rate governed by $\mu\lambda_{\min}$. A single $\mu$ has to serve both, so convergence along the cramped direction is roughly 28× slower. The bowl in that workshop is this ellipse; the "speed limit" is the long arrow, and the slow direction is the short one.

**One caution about interpreting the axes.** Eigenvectors of a covariance are the directions of maximal variance, which is not the same as the directions that are meaningful, independent, or physical. Rescale one coordinate — change units from volts to millivolts — and the covariance changes, so the axes rotate. PCA is **not scale-invariant**, which is why practitioners standardise features first, and why "the principal component" is a statement about your units as much as about your data.

Convergence of LMS in [Adaptive Filtering](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb) is governed by the eigenvalues of exactly this matrix $R$ — the $\mu < 2/\lambda_{max}$ bound is now a picture: the steepest axis of the bowl sets the speed limit.

---
### 🕐 Session 4 of 5 — *The SVD* (~40 min)
**Goal:** the decomposition that works for EVERY matrix; low-rank approximation and PCA.
**Builds on:** Session 3. &nbsp; **Feeds into:** Session 5 (matrix calculus).

---

<details>
<summary>🎓 <b>Teacher notes — Session 4: The SVD</b></summary>

**Timing (~40 min).** 8 min why eigendecomposition is not enough · 10 min rotate–stretch–rotate · 12 min low-rank approximation and Eckart–Young · 10 min PCA as SVD.

**Open by naming the limitation Session 3 left behind.** Eigendecomposition needs a square matrix, and behaves well only when it is symmetric. But most matrices in practice are neither: a data matrix is $n \times p$, a channel matrix is $m \times n$, a term-document matrix is rectangular by construction. Ask what "the eigenvectors of a $500\times2$ matrix" could possibly mean — the question is not hard, it is *ill-posed*. That gap is what the SVD fills.

**Then state the theorem in its strongest form, because it is genuinely remarkable.** $A = U\Sigma V^T$ exists for **every** matrix — real, complex, square, rectangular, singular, rank-deficient, all of it. No hypotheses. Every linear map, without exception, is **rotate → stretch along axes → rotate**. Nothing more complicated ever happens. Rooms that have struggled with the zoo of matrix types find this unifying, and it deserves to be delivered as the headline it is.

**Connect it to Session 3 rather than presenting it as new machinery.** $A^TA$ is symmetric and positive semi-definite, so the spectral theorem applies: its eigenvectors are $V$ and its eigenvalues are $\sigma_i^2$. The SVD is the spectral theorem applied to $A^TA$ and $AA^T$ simultaneously, with $U$ and $V$ linked by $Av_i = \sigma_iu_i$. Students who see this stop treating the SVD as a second, unrelated decomposition to memorise.

**Eckart–Young is the result that makes the SVD useful, so state it precisely.** Truncating to the top $r$ singular values gives the **provably best** rank-$r$ approximation in both Frobenius and spectral norm — not a good heuristic, the optimum over all rank-$r$ matrices. That word "best" is doing real work: it means no cleverer compression scheme of the same rank exists. Worth saying explicitly, since students often assume SVD truncation is one option among many.

**Run the image demo as a prediction exercise.** Ask how many of the 64 singular values are needed before showing the panels. Rank 1 already captures the dominant block structure; rank 3 adds the sinusoidal texture; by rank 8 the reconstruction is visually indistinguishable and holds **99.7%** of the energy. Then note the compression arithmetic: rank 8 of a $64\times64$ matrix stores $8(64+64+1) = 1032$ numbers against 4096, a 4× saving — and press on *why* it works. The image was built from a few structured pieces, so its true rank was always low; the SVD found that without being told.

**Be honest that the demo is favourable.** A photograph of foliage has a slowly-decaying spectrum and compresses poorly this way, which is exactly why JPEG uses a fixed DCT basis on small blocks rather than a per-image SVD. **Low-rank structure is a property of the matrix, not a guarantee of the method** — the same caveat as sparsity in Session 1, and worth repeating because students over-generalise both.

**Close on the PCA cell, which ties Sessions 3 and 4 together numerically.** Eigendecomposing the covariance and taking the SVD of the centered data give the *same* axes, with $\sigma_i^2/(n-1) = \lambda_i$ — and the printout confirms it to two decimals. Then say why anyone bothers with the SVD route: forming $X^TX$ squares the condition number, so for near-degenerate data the covariance path loses roughly half the available digits while the SVD path does not. Every serious PCA implementation takes the SVD of the data, never the eigendecomposition of the covariance.
</details>

## 5. Singular Value Decomposition

💡 **Intuition.** Eigendecomposition needs square (ideally symmetric) matrices. The SVD works for **any** matrix: $A = U\Sigma V^T$ says every linear map is *rotate → stretch along axes → rotate* — no exceptions. The singular values in $\Sigma$ rank the map's actions by importance, and chopping the small ones gives the **best possible** low-rank approximation (Eckart–Young). PCA, compression, denoising, and pseudo-inverses are all this one move.

In [8]:
# Low-rank approximation of a structured "image"
img = np.zeros((64, 64))
img[10:54, 10:54] = 1.0
yy, xx = np.mgrid[0:64, 0:64]
img += 0.5 * np.sin(2 * np.pi * xx / 16) * (yy > 32)
img += 0.05 * rng.standard_normal((64, 64))

U, s, Vt = np.linalg.svd(img)
fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("original (rank 64)")
for ax, r in zip(axes[1:], [1, 3, 8]):
    approx = U[:, :r] @ np.diag(s[:r]) @ Vt[:r]
    ax.imshow(approx, cmap="gray"); ax.set_title(f"rank {r}")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()
print("energy in top 8 of 64 singular values:", f"{(s[:8]**2).sum() / (s**2).sum():.1%}")

energy in top 8 of 64 singular values: 99.7%


/tmp/ipykernel_2010461/479179373.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Four panels, and the reconstruction quality climbs fast. Rank 1 already recovers the bright block against its dark surround. Rank 3 adds the sinusoidal texture in the lower half. By rank 8 the image is visually indistinguishable from the original, and the printout says why: **the top 8 of 64 singular values hold 99.7% of the energy**.

**Count the storage, because that is the claim being made.** The full matrix is $64 \times 64 = 4096$ numbers. A rank-8 truncation stores $8 \times (64 + 64 + 1) = 1032$ — a **4× compression** with 0.3% of the energy discarded. Rank 3 needs 387 numbers, a 10.6× compression, and still looks like the image.

**And this truncation is not merely good, it is optimal.** Eckart–Young says that keeping the top $r$ singular values gives the **best possible rank-$r$ approximation**, in both Frobenius and spectral norm, over *all* rank-$r$ matrices. No cleverer scheme of the same rank exists. That is a much stronger statement than "SVD compression works well", and it is the reason the SVD sits underneath so many methods — you are not choosing a heuristic, you are computing the answer.

**Now the honest caveat, because this image was built to succeed.** It is a rectangle plus one sinusoid plus a little noise — a handful of structured components, so its true rank was low before we started. The SVD did not create that structure, it **found** it. Feed in a photograph of foliage and the singular values decay slowly; rank 8 of 64 would look like a blur, and you would need most of the spectrum to reconstruct it. **Low rank is a property of the matrix, not a guarantee of the method** — precisely parallel to Session 1's point that sparsity is a property of the signal–basis pairing.

**That caveat explains a real engineering decision.** JPEG does *not* use a per-image SVD, despite Eckart–Young optimality, for two reasons visible here: the SVD costs $O(N^3)$ and its basis is data-dependent, so you must transmit $U$ and $V$ along with the coefficients. A fixed DCT basis on $8\times8$ blocks is suboptimal per block and enormously cheaper — no basis to send, $O(N\log N)$ to apply. **Optimality and practicality are different objectives**, and standards bodies chose the second.

**A note on the noise term, which is doing something useful.** The $0.05$ Gaussian noise added to the image is full-rank by construction — it spreads energy across all 64 singular values roughly evenly. Truncating to rank 8 therefore discards about $56/64$ of it. So the rank-8 panel is not just compressed, it is **denoised**: the same projection-onto-a-subspace argument as Session 2, now with the subspace chosen by the data rather than fixed in advance.

In [9]:
# PCA = SVD of centered data (relating S3 and S4)
Xc = X - X.mean(0)
U2, s2, Vt2 = np.linalg.svd(Xc, full_matrices=False)
print("PCA directions from SVD:\n", np.round(Vt2.T, 4))
print("same axes as covariance eigh (up to sign/order):\n", np.round(V, 4))
print("singular values² / (n−1) =", np.round(s2**2 / (len(Xc) - 1), 2), " vs eigenvalues", np.round(w[::-1], 2))

PCA directions from SVD:
 [[ 0.9939 -0.1105]
 [ 0.1105  0.9939]]
same axes as covariance eigh (up to sign/order):
 [[ 0.1105 -0.9939]
 [-0.9939 -0.1105]]
singular values² / (n−1) = [5.09 0.18]  vs eigenvalues [5.09 0.18]


**What just happened.** Two routes to the same answer. The SVD of the centered data returns directions $[0.9939, 0.1105]$ and $[-0.1105, 0.9939]$; the eigendecomposition of the covariance in Session 3 returned the same pair, reordered and sign-flipped. And the variances match exactly:

$$\frac{\sigma_i^2}{n-1} = [5.09,\ 0.18] \qquad\text{vs}\qquad \lambda_i = [5.09,\ 0.18]$$

**This is an identity, not a numerical coincidence.** For centered data, $C = \frac{X^TX}{n-1}$. Substituting $X = U\Sigma V^T$ gives $X^TX = V\Sigma^TU^TU\Sigma V^T = V\Sigma^2V^T$ — which is exactly the eigendecomposition of $C$, with eigenvectors $V$ and eigenvalues $\sigma_i^2/(n-1)$. **PCA and the SVD are the same computation**, and the printout is that algebra evaluated.

**The reordering and sign flips are expected, and worth flagging so nobody thinks the results disagree.** `eigh` returns eigenvalues in *ascending* order; `svd` returns singular values in *descending* order — hence the `[::-1]` in the comparison. And $-v$ is as valid an eigenvector as $v$, so any implementation may return either; there is no canonical sign. Code that compares eigenvectors across libraries must fix a convention or compare $|q^Tv|$.

**So why does anyone use the SVD route, if both give the same answer?** **Conditioning.** Forming $X^TX$ squares the condition number: if $X$ has $\kappa = 10^6$ — unremarkable for real data with correlated features — then $X^TX$ has $\kappa = 10^{12}$, and in double precision you have burned twelve of your sixteen digits before the eigensolver starts. The SVD works on $X$ directly and never forms that product, so it costs $\kappa$ rather than $\kappa^2$. **You lose half your digits by squaring, and you get them back for free by not squaring.**

**This is the same argument that made `lstsq` preferable to the normal equations in Session 2**, and it is the recurring numerical-linear-algebra lesson: *the mathematically equivalent formula and the numerically sound formula are frequently different formulas*. Every production PCA — scikit-learn's included — takes the SVD of the data matrix, never the eigendecomposition of the covariance, for precisely this reason.

**One more reason, worth a sentence.** When $p \gg n$ — 10,000 gene expressions on 200 patients — the covariance is $10{,}000 \times 10{,}000$ and mostly rank-deficient, while the SVD of the $200 \times 10{,}000$ data matrix needs only the 200 directions that actually carry information. Forming $C$ would mean building and decomposing a matrix 2,500× larger than necessary, with no extra information in it.

---
### 🕐 Session 5 of 5 — *Matrix Calculus* (~35 min)
**Goal:** differentiate through vectors and matrices — the notation backprop is written in.
**Builds on:** Sessions 2–4.

---

<details>
<summary>🎓 <b>Teacher notes — Session 5: Matrix Calculus</b></summary>

**Timing (~35 min).** 5 min deflating the topic · 10 min the two identities · 10 min least squares re-derived · 10 min numerical checking as a habit.

**Deflate the subject first, because the notation frightens people more than the mathematics warrants.** There is no new calculus here. $\nabla_x f$ is the vector whose $i$-th entry is $\partial f/\partial x_i$ — every entry is an ordinary partial derivative the room already knows. What is new is *bookkeeping*: keeping track of which index goes where without writing $n$ separate equations. Say this plainly; students who believe matrix calculus is a new subject spend weeks being intimidated by an accounting convention.

**Derive $\nabla_x(a^Tx) = a$ componentwise once, on the board.** $a^Tx = \sum_j a_jx_j$, so $\partial/\partial x_i = a_i$, so the gradient is $a$. Thirty seconds, and it converts the identity from something memorised into something obvious. Then note the analogy that makes it stick: this is the vector version of $\frac{d}{dx}(ax) = a$.

**Do the same for $\nabla_x(x^TSx) = 2Sx$ and be explicit that symmetry is a hypothesis.** Componentwise, $x^TSx = \sum_{j,k}S_{jk}x_jx_k$, and differentiating in $x_i$ picks up terms from both the $j = i$ and $k = i$ sums, giving $(S + S^T)x$. **Only when $S = S^T$ does that collapse to $2Sx$.** Students drop the symmetry condition constantly and then get factors of two wrong in Hessians. Note also the scalar analogy: $\frac{d}{dx}(ax^2) = 2ax$, with the same factor of 2 and the same source.

**Then re-derive least squares and let the room see two roads meet.** Expand $J(c) = \|Ac - x\|^2 = c^TA^TAc - 2x^TAc + x^Tx$, apply both identities, and get $\nabla_cJ = 2A^TAc - 2A^Tx$. Setting it to zero returns Session 2's normal equations **exactly**. Point out that Session 2 got there by pure geometry — "the residual must be orthogonal to the subspace" — with no derivatives at all. Optimality and orthogonality really are the same statement, and now the room has seen both derivations land on the same equation.

**Make the numerical check a professional habit, not a demo.** The central difference $\frac{f(x+\varepsilon e_i) - f(x-\varepsilon e_i)}{2\varepsilon}$ has error $O(\varepsilon^2)$, versus $O(\varepsilon)$ for the one-sided version — worth showing, because it is the reason the code uses it. Frame the habit correctly: **never trust a hand-derived gradient you have not checked numerically.** It is five lines, it runs in milliseconds, and an unchecked sign error in a gradient produces a model that trains slowly and silently rather than one that crashes. Every serious autodiff library ships a `gradcheck` for exactly this reason, and the [ANN workshop](../../Intro_Mach_Learn/Intro_ANN/Intro_ANN.ipynb) already made the same point about backprop.

**If time allows, mention the $\varepsilon$ tradeoff, since students will otherwise reach for $10^{-12}$.** Too large and truncation error dominates; too small and floating-point cancellation in the numerator destroys the answer. Around $10^{-6}$ balances them for double precision, which is why the code chose it. This is a genuine numerical-analysis lesson hiding in a five-line helper.

**Close the workshop by re-reading the opening sentence and checking off all five clauses.** Signals are vectors (S1); bases make hard signals sparse (S1); projections are optimal approximations (S2); eigenvectors are a matrix's natural axes (S3); the SVD does it for every matrix (S4); two gradient identities unlock the optimization (S5). Then point forward: [Optimization](../Optimization/Optimization.ipynb) is what to *do* with those gradients, and [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) is the star change-of-basis. Every other workshop in the curriculum now has its algebra.
</details>

## 6. Gradients of Vector Functions

💡 **Intuition.** Matrix calculus is scalar calculus plus bookkeeping: the gradient of a scalar with respect to a vector is just the vector of partials, and the two identities below cover 90% of everything in this curriculum. When in doubt, *check numerically* — the habit the [ANN workshop](../../Intro_Mach_Learn/Intro_ANN/Intro_ANN.ipynb) already taught you.

**The two workhorses** (with $S$ symmetric):

$$\nabla_x \, (a^T x) = a \qquad \nabla_x \, (x^T S x) = 2 S x$$

**Application — least squares in one line.** $J(c) = \|Ac - x\|^2 = c^T A^T A c - 2 x^T A c + x^T x$, so $\nabla_c J = 2 A^T A c - 2 A^T x$. Set to zero → the normal equations of Session 2. The LMS update of [Adaptive Filtering](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb) is gradient descent on exactly this $J$.

In [10]:
# Verify both identities numerically — never trust a derivative you haven't checked
def num_grad(f, x, eps=1e-6):
    g = np.zeros_like(x)
    for i in range(len(x)):
        e = np.zeros_like(x); e[i] = eps
        g[i] = (f(x + e) - f(x - e)) / (2 * eps)
    return g

x0 = rng.standard_normal(5)
a = rng.standard_normal(5)
M = rng.standard_normal((5, 5)); S5 = M + M.T

print("‖∇(aᵀx) − a‖          =", np.abs(num_grad(lambda x: a @ x, x0) - a).max().round(9))
print("‖∇(xᵀSx) − 2Sx‖       =", np.abs(num_grad(lambda x: x @ S5 @ x, x0) - 2 * S5 @ x0).max().round(6))

A2 = rng.standard_normal((8, 5)); b2 = rng.standard_normal(8)
g_analytic = 2 * A2.T @ (A2 @ x0 - b2)
g_numeric  = num_grad(lambda c: np.sum((A2 @ c - b2)**2), x0)
print("‖∇‖Ac−b‖² check‖      =", np.abs(g_analytic - g_numeric).max().round(6))

‖∇(aᵀx) − a‖          = 0.0
‖∇(xᵀSx) − 2Sx‖       = 0.0
‖∇‖Ac−b‖² check‖      = 0.0


**What just happened.** All three checks print **0.0** — the hand-derived gradients agree with central differences to the rounding precision requested. $\nabla(a^Tx) = a$, $\nabla(x^TSx) = 2Sx$, and the least-squares gradient $2A^T(Ac - b)$ all survive contact with a numerical derivative.

**Note the middle identity's hypothesis, since it is the one students drop.** Differentiating $x^TSx = \sum_{j,k}S_{jk}x_jx_k$ componentwise picks up terms from both index positions, giving $(S + S^T)x$ — which collapses to $2Sx$ **only because $S$ is symmetric**. The code enforces this with `S5 = M + M.T`; run it with a general $M$ and the check fails by exactly the antisymmetric part. Every misplaced factor of 2 in a Hessian traces back to this line.

**The third check is the session's punchline.** $\nabla_c\|Ac - b\|^2 = 2A^T(Ac - b)$; set it to zero and you have $A^TAc = A^Tb$ — Session 2's normal equations, derived here by calculus rather than by geometry. Two entirely different arguments, one from "the residual must be orthogonal to the subspace" and one from "the derivative must vanish", arriving at the identical equation. **Optimality and orthogonality are the same statement**, and this cell is the receipt.

**Read the zeros carefully, because they are `.round()` of something nonzero.** The central difference has truncation error $O(\varepsilon^2)$ and cancellation error $O(\varepsilon_{\text{mach}}/\varepsilon)$; at $\varepsilon = 10^{-6}$ these balance around $10^{-10}$. So the true discrepancies are near $10^{-10}$, displayed as 0.0 after rounding to 9 and 6 places. That is agreement to the limit of what finite differences can resolve — which is the strongest claim this method can make, and a different claim from exactness.

**Why $\varepsilon = 10^{-6}$ and not smaller, since students always try.** Shrink to $10^{-12}$ and the numerator becomes the difference of two nearly identical doubles: catastrophic cancellation destroys the significant digits, and the "check" starts reporting errors of order 1 for a perfectly correct gradient. Too large and truncation dominates instead. Around $\sqrt[3]{\varepsilon_{\text{mach}}} \approx 10^{-5}$ is the sweet spot for central differences, and a five-line helper has a genuine numerical-analysis decision buried in it.

**The habit is the real deliverable here, not the identities.** A wrong gradient does not crash — it trains slowly, converges to the wrong place, or plateaus mysteriously, and it can cost days to find. A numerical check costs milliseconds and catches sign errors, transposes, and missing factors of 2 immediately. Every serious autodiff framework ships a `gradcheck` for exactly this reason. **Never trust a derivative you have not checked**, and now you know the five lines that check one.

## 7. Conclusion

Signals are vectors; bases make hard signals sparse; projections are optimal approximations; eigenvectors are a matrix's natural axes; the SVD does it for every matrix; and two gradient identities unlock all the optimization. Every other workshop now has its algebra.

---
## Where next

- [Optimization](../Optimization/Optimization.ipynb) — what to *do* with those gradients.
- [Foundations of Signal Processing](../../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) — the Fourier basis as the star change-of-basis.
- [Statistical Signal Processing](../../Intro_DSP/Statistical_Signal_Processing.ipynb) — covariance eigenstructure at work.